# The Count of Monte Cristo — LLM-Built Character Knowledge Graph

Agentic pipeline that reads the full novel, extracts character-level causal events and entity/relationship triplets via an LLM (Mistral `ministral-8b-latest`), resolves character aliases to canonical identities, assembles a knowledge graph, and analyzes it with spectral graph theory (Laplacian eigenmaps, diffusion maps).

**Pipeline stages:**
1. Chunked causal-event extraction from raw text (sliding window + LLM)
2. Character profile construction from extracted events
3. LightRAG-style entity/relationship triplet extraction
4. Character name deduplication (string similarity + manual alias maps)
5. Knowledge graph assembly
6. Final deep merge + hierarchical attribute agents (spatial / attribute / causal) per character
7. Star-graph fusion and visualization

**Known limitations (read before reusing this pipeline on another book):**
- Character alias resolution relies on large, hand-curated dictionaries (`MASTER_MAP`, `REPAIR_MAP`, `alias_map`) built by manually inspecting LLM output for *this specific novel*. It is not a generalized entity-resolution system — swapping in a different book will require rebuilding these maps from scratch.
- There is no ground-truth evaluation of extracted relationships (no sampled precision/recall check against the text).
- A few path/filename inconsistencies from iterative development are flagged inline below where found — verify before rerunning end-to-end.

## 1. Setup

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
pip install mistralai

## 2. Causal Skeleton Extraction
Sliding-window LLM extraction of "causal anchor" events per character, run in parts to work around long-running sessions, with a gap-recovery pass for windows that failed partway through.

In [ ]:
import os
import json
import time
from mistralai import Mistral
from kaggle_secrets import UserSecretsClient #

# --- 1. SECURE CONFIGURATION & PATHS ---
user_secrets = UserSecretsClient() 
MISTRAL_API_KEY = user_secrets.get_secret("MISTRAL_API_KEY") #

MODEL_ID = "ministral-8b-latest" #
BOOK_PATH = "/kaggle/input/books-iit-kgp/The Count of Monte Cristo.txt"

# Creating the requested folder structure
OUTPUT_FOLDER = "the_count_of_monte_cristo"
if not os.path.exists(OUTPUT_FOLDER):
    os.makedirs(OUTPUT_FOLDER)
    
OUTPUT_FILE = os.path.join(OUTPUT_FOLDER, "monte_cristo_chunk.jsonl")

# --- 2. DYNAMIC CONTROL VARIABLES ---
PARTIAL_PROCESSING = False  # Set to True if you only want the first 5 chunks
IDEAL_WINDOW_SIZE = 3000   #
STRIDE = 2800              #

client = Mistral(api_key=MISTRAL_API_KEY)

# --- 3. THE REFINED AGENTIC PROMPT ---
SYSTEM_PROMPT = """You are a Narrative Causal Analyst. Your task is to process a long-form novel and extract a "Causal Skeleton" to verify a character's backstory. 

### GOAL:
Analyze the provided text window and determine if it contains a "Causal Anchor." If it does, create a structured summary chunk. If the text is purely descriptive or irrelevant to the character's logical state, ignore it.

### CRITERIA FOR A NEW CHUNK (State Transitions):
1. IDENTITY/ORIGIN: Facts about family, social class, or birth.
2. FORMATIVE EXPERIENCE: Events establishing a fear, belief, or ambition.
3. PHYSICAL CONSTRAINT: Injuries, aging, or irreversible changes in state.
4. CAUSAL PIVOT: A major action that rules out alternative futures (e.g., moving to a new city, taking a job, a death).
5. WORLD RULES: Information defining what is possible or impossible in this story's universe.

### OUTPUT FORMAT:
You must output ONLY a valid JSON object. Do not include conversational text.
{
  "boundary_detected": true/false,
  "verbatim_anchor": "The exact sentence from the text that establishes the constraint",
  "causal_summary": "A 1-sentence explanation of the logical state change",
  "character_state": {
    "name": "Character Name",
    "attribute": "e.g., Blind, Wealthy, Exiled",
    "location": "Current location"
  },
  "logic_type": "Identity/Constraint/Pivot/Rule"
}"""

def call_analyst(text_window):
    """Executes API call with hardware-enforced JSON formatting."""
    try:
        response = client.chat.complete(
            model=MODEL_ID,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"### TEXT TO ANALYZE:\n{text_window}"}
            ],
            temperature=0.1,       # Low for deterministic reasoning
            max_tokens=2048,      # Sufficient for detailed state descriptions
            response_format={"type": "json_object"} # Hard-enforces valid JSON output
        )
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        print(f"Error in window analysis: {e}")
        return None

# --- 4. THE SEQUENTIAL EXECUTION ENGINE ---
def run_full_extraction():
    if not os.path.exists(BOOK_PATH):
        print(f"Error: {BOOK_PATH} not found.")
        return

    with open(BOOK_PATH, 'r', encoding='utf-8') as f:
        full_text = f.read()

    current_pos = 0
    chunks_found = 0
    
    print(f"--- STARTING CAUSAL SKELETON EXTRACTION ---")
    print(f"Output: {OUTPUT_FILE}")
    
    

    with open(OUTPUT_FILE, 'w') as out_f:
        while current_pos < len(full_text):
            end_pos = min(current_pos + IDEAL_WINDOW_SIZE, len(full_text))
            window_text = full_text[current_pos:end_pos]
            
            res = call_analyst(window_text)
            
            if res and res.get("boundary_detected"):
                chunks_found += 1
                out_f.write(json.dumps(res) + "\n")
                print(f"[CHUNK #{chunks_found}] Type: {res.get('logic_type')} | Name: {res.get('character_state', {}).get('name')}")
                
                if PARTIAL_PROCESSING and chunks_found >= 5:
                    break
            
            current_pos += STRIDE
            if end_pos == len(full_text): break

    print(f"\nExtraction complete. Saved {chunks_found} chunks to {OUTPUT_FILE}")

run_full_extraction()

In [ ]:
import os
import json
import time
from mistralai import Mistral
from kaggle_secrets import UserSecretsClient

# --- 1. SECURE CONFIGURATION & PATHS ---
user_secrets = UserSecretsClient() 
MISTRAL_API_KEY = user_secrets.get_secret("MISTRAL_API_KEY") 

MODEL_ID = "ministral-8b-latest"
BOOK_PATH = "/kaggle/input/books-iit-kgp/The Count of Monte Cristo.txt"

# Path setup for Part 2
OUTPUT_FOLDER = "the_count_of_monte_cristo"
if not os.path.exists(OUTPUT_FOLDER):
    os.makedirs(OUTPUT_FOLDER)
    
# Updated filename as requested
OUTPUT_FILE = os.path.join(OUTPUT_FOLDER, "monte_cristo_chunks_part 2.jsonl")

# --- 2. DYNAMIC CONTROL VARIABLES ---
IDEAL_WINDOW_SIZE = 3000   
STRIDE = 2800              
# Resuming exactly where the first 121 chunks (12.8%) left off
RESUME_POS = 338800 
PARTIAL_PROCESSING = False 

client = Mistral(api_key=MISTRAL_API_KEY)

# --- 3. THE REFINED AGENTIC PROMPT ---
SYSTEM_PROMPT = """You are a Narrative Causal Analyst. Your task is to process a long-form novel and extract a "Causal Skeleton" to verify a character's backstory. 

### GOAL:
Analyze the provided text window and determine if it contains a "Causal Anchor." If it does, create a structured summary chunk. If the text is purely descriptive or irrelevant to the character's logical state, ignore it.

### CRITERIA FOR A NEW CHUNK (State Transitions):
1. IDENTITY/ORIGIN: Facts about family, social class, or birth.
2. FORMATIVE EXPERIENCE: Events establishing a fear, belief, or ambition.
3. PHYSICAL CONSTRAINT: Injuries, aging, or irreversible changes in state.
4. CAUSAL PIVOT: A major action that rules out alternative futures (e.g., moving to a new city, taking a job, a death).
5. WORLD RULES: Information defining what is possible or impossible in this story's universe.

### OUTPUT FORMAT:
You must output ONLY a valid JSON object. Do not include conversational text.
{
  "boundary_detected": true/false,
  "verbatim_anchor": "The exact sentence from the text that establishes the constraint",
  "causal_summary": "A 1-sentence explanation of the logical state change",
  "character_state": {
    "name": "Character Name",
    "attribute": "e.g., Blind, Wealthy, Exiled",
    "location": "Current location"
  },
  "logic_type": "Identity/Constraint/Pivot/Rule"""

def call_analyst(text_window):
    """Executes API call and handles List vs Dictionary outputs to prevent AttributeErrors."""
    try:
        response = client.chat.complete(
            model=MODEL_ID,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"### TEXT TO ANALYZE:\n{text_window}"}
            ],
            temperature=0.1,
            max_tokens=2048,
            response_format={"type": "json_object"} 
        )
        
        parsed_data = json.loads(response.choices[0].message.content)
        
        # --- THE FIX: Handle List responses to avoid 'AttributeError' ---
        if isinstance(parsed_data, list):
            # Take the first identified anchor if the model returns multiple
            return parsed_data[0] if len(parsed_data) > 0 else None
        return parsed_data
        
    except Exception as e:
        print(f"Error in window analysis: {e}")
        return None

# --- 4. THE RESUMED EXECUTION ENGINE ---
def run_part_2_extraction():
    if not os.path.exists(BOOK_PATH):
        print(f"Error: {BOOK_PATH} not found.")
        return

    with open(BOOK_PATH, 'r', encoding='utf-8') as f:
        full_text = f.read()

    current_pos = RESUME_POS
    new_chunks_found = 0
    
    print(f"--- STARTING PART 2 EXTRACTION AT POS {current_pos:,} ---")
    print(f"Saving to: {OUTPUT_FILE}")

    # Using 'w' here because this is a new file for Part 2
    with open(OUTPUT_FILE, 'w') as out_f:
        while current_pos < len(full_text):
            end_pos = min(current_pos + IDEAL_WINDOW_SIZE, len(full_text))
            window_text = full_text[current_pos:end_pos]
            
            res = call_analyst(window_text)
            
            # --- THE SAFETY CHECK: Ensure res is a dict before using .get() ---
            if res and isinstance(res, dict) and res.get("boundary_detected"):
                new_chunks_found += 1
                out_f.write(json.dumps(res) + "\n")
                
                # Progress tracking
                progress = (current_pos / len(full_text)) * 100
                print(f"[{progress:.2f}%] NEW CHUNK #{new_chunks_found}: {res.get('logic_type')}")
                
                if PARTIAL_PROCESSING and new_chunks_found >= 5:
                    break
            
            current_pos += STRIDE
            if end_pos == len(full_text): break

    print(f"\nPart 2 complete. Saved {new_chunks_found} new chunks.")

run_part_2_extraction()

In [ ]:
# --- CONFIGURATION ---
BOOK_PATH = "/kaggle/input/books-iit-kgp/The Count of Monte Cristo.txt"
TOTAL_CHARS = 2646614
STRIDE = 2800
WINDOW = 3000
OUTPUT_FILE = "skipped_monte_cristo.txt"

# --- CALCULATING THE GAPS ---
# Gap 1 starts 1 stride after Chunk #181 (35.12%) and covers 4 failed windows
pos_181 = int(TOTAL_CHARS * 0.3512)
start_gap_1 = pos_181 + STRIDE
# 4 windows of 2800 stride + the 3000 char window size of the last failed one
end_gap_1 = start_gap_1 + (3 * STRIDE) + WINDOW 

# Gap 2 starts 1 stride after Chunk #182 (36.18%) and covers 4 failed windows
pos_182 = int(TOTAL_CHARS * 0.3618)
start_gap_2 = pos_182 + STRIDE
end_gap_2 = start_gap_2 + (3 * STRIDE) + WINDOW

# --- EXTRACTION ---
try:
    with open(BOOK_PATH, 'r', encoding='utf-8') as f:
        book_text = f.read()

    # Slicing the exact skipped ranges
    skipped_text_1 = book_text[start_gap_1:end_gap_1]
    skipped_text_2 = book_text[start_gap_2:end_gap_2]

    # Saving to the requested file
    with open(OUTPUT_FILE, "w", encoding='utf-8') as out:
        out.write(f"=== SKIPPED TEXT AFTER CHUNK #181 (Range: {start_gap_1} - {end_gap_1}) ===\n")
        out.write(skipped_text_1)
        out.write("\n\n" + "="*50 + "\n\n")
        out.write(f"=== SKIPPED TEXT AFTER CHUNK #182 (Range: {start_gap_2} - {end_gap_2}) ===\n")
        out.write(skipped_text_2)

    print(f"Extraction successful! Missing text saved to {OUTPUT_FILE}")
    print(f"Total characters recovered: {len(skipped_text_1) + len(skipped_text_2):,}")

except FileNotFoundError:
    print(f"Error: Could not find the book at {BOOK_PATH}")
except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
import os
import json
import time
from mistralai import Mistral
from kaggle_secrets import UserSecretsClient

# --- 1. SECURE CONFIGURATION ---
user_secrets = UserSecretsClient() 
MISTRAL_API_KEY = user_secrets.get_secret("MISTRAL_API_KEY") 
client = Mistral(api_key=MISTRAL_API_KEY)

MODEL_ID = "ministral-8b-latest"
SKIPPED_TXT_PATH = "/kaggle/working/skipped_monte_cristo.txt"

# Output files for the two gaps
PATCH_1_JSON = "skipped_json_part_1.jsonl"
PATCH_2_JSON = "skipped_json_part_2.jsonl"

# Window parameters used in your previous successful runs
WINDOW_SIZE = 3000
STRIDE = 2800

# --- 2. INTEGRATED AGENTIC PROMPT ---
SYSTEM_PROMPT = """You are a Narrative Causal Analyst. Your task is to process a long-form novel and extract a "Causal Skeleton" to verify a character's backstory. 

### GOAL:
Analyze the provided text window and determine if it contains a "Causal Anchor." If it does, create a structured summary chunk. If the text is purely descriptive or irrelevant to the character's logical state, ignore it.

### CRITERIA FOR A NEW CHUNK (State Transitions):
1. IDENTITY/ORIGIN: Facts about family, social class, or birth.
2. FORMATIVE EXPERIENCE: Events establishing a fear, belief, or ambition.
3. PHYSICAL CONSTRAINT: Injuries, aging, or irreversible changes in state.
4. CAUSAL PIVOT: A major action that rules out alternative futures (e.g., moving to a new city, taking a job, a death).
5. WORLD RULES: Information defining what is possible or impossible in this story's universe.

### OUTPUT FORMAT:
You must output ONLY a valid JSON object. Do not include conversational text.
{
  "boundary_detected": true/false,
  "verbatim_anchor": "The exact sentence from the text that establishes the constraint",
  "causal_summary": "A 1-sentence explanation of the logical state change",
  "character_state": {
    "name": "Character Name",
    "attribute": "e.g., Blind, Wealthy, Exiled",
    "location": "Current location"
  },
  "logic_type": "Identity/Constraint/Pivot/Rule"
}"""

# --- 3. ROBUST ANALYST (With 429 Retry Logic & List-Fix) ---
def call_analyst_robust(text_window, max_retries=5):
    """Executes API call with exponential backoff for 429 errors and handles List outputs."""
    retries = 0
    wait_time = 4 
    while retries < max_retries:
        try:
            response = client.chat.complete(
                model=MODEL_ID,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": f"### TEXT TO ANALYZE:\n{text_window}"}
                ],
                temperature=0.1,
                response_format={"type": "json_object"}
            )
            parsed_data = json.loads(response.choices[0].message.content)
            
            # THE FIX: If the model sends a list, grab the first box to avoid AttributeError 
            if isinstance(parsed_data, list):
                return parsed_data[0] if len(parsed_data) > 0 else None
            return parsed_data
            
        except Exception as e:
            if "429" in str(e) or "rate_limited" in str(e).lower():
                print(f"Rate limit (429) hit. Waiting {wait_time}s... ({retries+1}/{max_retries})")
                time.sleep(wait_time)
                retries += 1
                wait_time *= 2  # Exponential backoff 
            else:
                print(f"Permanent Error: {e}")
                return None
    return None

# --- 4. DATA PARSING & PATCHING ENGINE ---
def process_skipped_segments():
    if not os.path.exists(SKIPPED_TXT_PATH):
        print(f"Error: {SKIPPED_TXT_PATH} not found.")
        return

    with open(SKIPPED_TXT_PATH, 'r', encoding='utf-8') as f:
        raw_content = f.read()

    # Split using the equals-sign separator
    parts = raw_content.split("==================================================")
    
    # Extract raw text by stripping the marker headers 
    # Part 1: Chars between marker 181 and the separator
    text_1 = parts[0].split("=== SKIPPED TEXT AFTER CHUNK #181")[1].split("===\n", 1)[1].strip()
    # Part 2: Chars after marker 182
    text_2 = parts[1].split("=== SKIPPED TEXT AFTER CHUNK #182")[1].split("===\n", 1)[1].strip()

    gap_tasks = [(text_1, PATCH_1_JSON), (text_2, PATCH_2_JSON)]

    for text_block, output_file in gap_tasks:
        print(f"\n--- PATCHING: {output_file} ({len(text_block)} characters) ---")
        current_pos = 0
        chunks_found = 0
        
        with open(output_file, 'w') as out_f:
            while current_pos < len(text_block):
                # Standard windowing to ensure 200 char overlap 
                end_pos = min(current_pos + WINDOW_SIZE, len(text_block))
                window = text_block[current_pos:end_pos]
                
                res = call_analyst_robust(window)
                
                if res and isinstance(res, dict) and res.get("boundary_detected"):
                    chunks_found += 1
                    out_f.write(json.dumps(res) + "\n")
                    print(f"Patching Chunk #{chunks_found}: {res.get('logic_type')}")
                
                current_pos += STRIDE
                if end_pos == len(text_block): break
                time.sleep(1) # Extra stability for rate limits

    print(f"\nGap Recovery Complete. Patches saved to JSONL files.")

process_skipped_segments()

In [ ]:
import json
import os

# --- 1. FILE PATHS ---
PART_1_FILE = "/kaggle/working/the_count_of_monte_cristo/monte_cristo_chunk.jsonl"
PART_2_FILE = "/kaggle/working/the_count_of_monte_cristo/monte_cristo_chunks_part 2.jsonl"
PATCH_2_FILE = "/kaggle/working/skipped_json_part_2.jsonl"
MASTER_OUTPUT = "/kaggle/working/the_count_of_monte_cristo/monte_cristo_master_skeleton.jsonl"

def merge_causal_skeleton():
    master_chunks = []

    # A. LOAD PART 1 (The first 121 chunks)
    print("Loading Part 1...")
    if os.path.exists(PART_1_FILE):
        with open(PART_1_FILE, 'r') as f:
            for line in f:
                master_chunks.append(json.loads(line))
    
    # B. LOAD PART 2 WITH PATCH INSERTION
    print("Loading Part 2 and inserting patches...")
    if os.path.exists(PART_2_FILE):
        p2_counter = 0
        with open(PART_2_FILE, 'r') as f:
            for line in f:
                p2_counter += 1
                master_chunks.append(json.loads(line))
                
                # Splicing Logic: Insert the skipped chunks after Part 2's Chunk #182
                if p2_counter == 182 and os.path.exists(PATCH_2_FILE):
                    print(f"--- Splicing Patch 2 after Chunk #{p2_counter} ---")
                    with open(PATCH_2_FILE, 'r') as pf:
                        for pline in pf:
                            master_chunks.append(json.loads(pline))

    # C. SAVE MASTER FILE
    with open(MASTER_OUTPUT, 'w') as out_f:
        for chunk in master_chunks:
            # Standardize logic_type to Title Case during merge
            chunk['logic_type'] = chunk.get('logic_type', '').strip().title()
            out_f.write(json.dumps(chunk) + "\n")

    print(f"\nSUCCESS: Master Skeleton created with {len(master_chunks)} chunks.")
    print(f"Final File: {MASTER_OUTPUT}")

merge_causal_skeleton()

## 3. Character Profile Construction
⚠️ **Path note:** `MASTER_SKELETON` below points at `castaway-chunks/castaways_chunks_final.jsonl`, which looks like a leftover path from a different book/project rather than the `monte_cristo_master_skeleton.jsonl` produced above — verify this before rerunning. Similarly, the save path in the next cell (`casteaway_character_profiles.json`) doesn't match the load path the cell after it expects (`monte_cristo_character_profiles.json`) — fix one or the other before relying on this step.

In [ ]:
import json
from collections import defaultdict

MASTER_SKELETON = "/kaggle/input/casteaway-chunks/castaways_chunks_final.jsonl"

def build_character_profiles():
    # Using a dictionary to store the "latest" state for each character
    profiles = defaultdict(lambda: {"attributes": [], "locations": [], "logic_timeline": []})

    with open(MASTER_SKELETON, 'r') as f:
        for line in f:
            chunk = json.loads(line)
            char_data = chunk.get("character_state", {})
            name = char_data.get("name")
            
            if name and name != "Character Name":
                # Add unique attributes and locations as they evolve
                if char_data.get("attribute") not in profiles[name]["attributes"]:
                    profiles[name]["attributes"].append(char_data.get("attribute"))
                
                profiles[name]["locations"].append(char_data.get("location"))
                
                # Build a timeline of their logical transitions
                profiles[name]["logic_timeline"].append({
                    "type": chunk.get("logic_type"),
                    "summary": chunk.get("causal_summary")
                })

    # Save the profiles as a summary JSON
    with open("character_profiles.json", "w") as out:
        json.dump(profiles, out, indent=2)
    
    print(f"Profiles built for {len(profiles)} unique entities.")
    return profiles

profiles = build_character_profiles()

In [ ]:
import json
import os

# --- CONFIGURATION ---
PROFILE_OUTPUT_PATH = "/kaggle/working/the_count_of_monte_cristo/casteaway_character_profiles.json"

def save_profiles(profile_data):
    """Saves the aggregated character profiles to a JSON file."""
    try:
        # Ensure the output directory exists
        os.makedirs(os.path.dirname(PROFILE_OUTPUT_PATH), exist_ok=True)
        
        with open(PROFILE_OUTPUT_PATH, "w", encoding='utf-8') as out:
            # indent=2 makes the file human-readable for your review
            json.dump(profile_data, out, indent=2)
            
        print(f"Successfully saved {len(profile_data)} character profiles to: {PROFILE_OUTPUT_PATH}")
    except Exception as e:
        print(f"Error saving profiles: {e}")

# Call the function with your existing 'profiles' variable
save_profiles(profiles)

In [ ]:
import json

PROFILE_PATH = "/kaggle/working/the_count_of_monte_cristo/monte_cristo_character_profiles.json"

# Load existing data
with open(PROFILE_PATH, 'r', encoding='utf-8') as f:
    data = json.load(f)

# Save it back with Unicode characters preserved
with open(PROFILE_PATH, 'w', encoding='utf-8') as f:
    # ensure_ascii=False is the key fix here
    json.dump(data, f, indent=2, ensure_ascii=False)

print("Character profiles cleaned and updated!")

In [ ]:
import os
import json

# --- 1. CONFIGURATION ---
MASTER_PATH = "/kaggle/working/the_count_of_monte_cristo/monte_cristo_master_skeleton.jsonl"
# New permanent directory
FIXED_DIR = "/kaggle/working/fixed"
FIXED_PATH = os.path.join(FIXED_DIR, "monte_cristo_master_skeleton_fixed.jsonl")

# --- 2. CREATE PERMANENT FOLDER ---
# exist_ok=True ensures no error is raised if the folder already exists
os.makedirs(FIXED_DIR, exist_ok=True)

# --- 3. CLEAN AND SAVE ---
print(f"Cleaning master skeleton and saving to {FIXED_PATH}...")

try:
    with open(MASTER_PATH, 'r', encoding='utf-8') as infile, \
         open(FIXED_PATH, 'w', encoding='utf-8') as outfile:
        
        for line in infile:
            obj = json.loads(line)
            # ensure_ascii=False preserves non-ASCII characters like 'è'
            outfile.write(json.dumps(obj, ensure_ascii=False) + "\n")

    print("Success! Fixed master skeleton is now in the 'fixed' folder.")

except FileNotFoundError:
    print(f"Error: The source file was not found at {MASTER_PATH}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

## 4. Knowledge Graph Triplet Extraction
LightRAG-style entity/relationship extraction prompt, run as a quick 5-chunk test before the full pass.

In [ ]:
import os
import json
import time
from mistralai import Mistral
from kaggle_secrets import UserSecretsClient

# --- 1. CONFIGURATION ---
user_secrets = UserSecretsClient() 
MISTRAL_API_KEY = user_secrets.get_secret("MISTRAL_API_KEY") 
client = Mistral(api_key=MISTRAL_API_KEY)

MODEL_ID = "ministral-8b-latest"
INPUT_FILE = "/kaggle/working/fixed/monte_cristo_master_skeleton_fixed.jsonl"
OUTPUT_FOLDER = "/kaggle/working/Knowledge graph"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# Test file name
TEST_OUTPUT_FILE = os.path.join(OUTPUT_FOLDER, "monte_cristo_test_run.md")

# --- 2. THE OFFICIAL LIGHTRAG PROMPT ---
GRAPH_CONSTRUCT_PROMPT = """-Goal-
Given a text document and a list of entity types, identify all entities of those types and all relationships among them.

-Steps-
1. Identify all entities. Extract: entity_name, entity_type, and entity_description.
Format: ("entity"<|><entity_name><|><entity_type><|><entity_description>)

2. Identify related pairs. Extract: relationship_description, relationship_strength, and relationship_keywords.
Format: ("relationship"<|><source_entity><|><target_entity><|><relationship_description><|><relationship_keywords><|><relationship_strength>)

3. Identify high-level key words for themes.
Format: ("content_keywords"<|><high_level_keywords>)

4. Return output as a single list. Use "##" as the list delimiter.
5. When finished, output <COMPLETE>"""

# --- 3. TEST EXECUTION ENGINE ---
def run_test_extraction():
    if not os.path.exists(INPUT_FILE):
        print(f"Error: {INPUT_FILE} not found.")
        return

    print(f"--- STARTING TEST RUN (5 CHUNKS) ---")
    print(f"Saving to: {TEST_OUTPUT_FILE}\n")

    with open(INPUT_FILE, 'r', encoding='utf-8') as f, \
         open(TEST_OUTPUT_FILE, 'w', encoding='utf-8') as out_f:
        
        out_f.write("# Monte Cristo TEST RUN: Raw Extraction\n")
        out_f.write(f"Test Run Date: {time.strftime('%Y-%m-%d %H:%M:%S')}\n\n")

        for i, line in enumerate(f):
            if i >= 5: break # LIMIT TO 5 CHUNKS FOR TEST
            
            chunk = json.loads(line)
            summary = chunk.get("causal_summary", "")
            
            try:
                response = client.chat.complete(
                    model=MODEL_ID,
                    messages=[
                        {"role": "system", "content": GRAPH_CONSTRUCT_PROMPT},
                        {"role": "user", "content": f"Entity_types: [person, organization, geo, event]\nText: {summary}"}
                    ],
                    temperature=0.1
                )
                raw_data = response.choices[0].message.content
                
                # Write to Markdown
                out_f.write(f"## Chunk #{i}\n")
                out_f.write(f"**Causal Summary:** {summary}\n\n")
                out_f.write("### Raw LLM Output:\n")
                out_f.write("```text\n")
                out_f.write(raw_data)
                out_f.write("\n```\n\n")
                out_f.write("---\n\n")
                
                print(f"Chunk #{i} processed.")
                out_f.flush()
                
            except Exception as e:
                print(f"Error at Chunk {i}: {e}")
            
            time.sleep(1)

    print(f"\nTest run complete. Please check the file: {TEST_OUTPUT_FILE}")

run_test_extraction()

In [ ]:
import os
import json
import time
from mistralai import Mistral
from kaggle_secrets import UserSecretsClient

# --- 1. CONFIGURATION ---
user_secrets = UserSecretsClient() 
MISTRAL_API_KEY = user_secrets.get_secret("MISTRAL_API_KEY") 
client = Mistral(api_key=MISTRAL_API_KEY)

MODEL_ID = "ministral-8b-latest"
INPUT_FILE = "/kaggle/working/fixed/monte_cristo_master_skeleton_fixed.jsonl"
OUTPUT_FOLDER = "/kaggle/working/Knowledge graph"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# Extension changed to .md as requested
OUTPUT_FILE = os.path.join(OUTPUT_FOLDER, "monte_cristo_light_KG.md")

# --- 2. THE OFFICIAL LIGHTRAG PROMPT ---
GRAPH_CONSTRUCT_PROMPT = """-Goal-
Given a text document and a list of entity types, identify all entities of those types and all relationships among them.

-Steps-
1. Identify all entities. Extract: entity_name (capitalized), entity_type (organization, person, geo, event), and entity_description.
Format: ("entity"<|><entity_name><|><entity_type><|><entity_description>)

2. Identify related pairs (source_entity, target_entity). Extract: relationship_description, relationship_strength (1-10), and relationship_keywords.
Format: ("relationship"<|><source_entity><|><target_entity><|><relationship_description><|><relationship_keywords><|><relationship_strength>)

3. Identify high-level key words that summarize the main concepts, themes, or topics of the entire text.
Format: ("content_keywords"<|><high_level_keywords>)

4. Return output as a single list. Use "##" as the list delimiter.
5. When finished, output <COMPLETE>"""

# --- 3. EXECUTION ENGINE (RAW MD MODE) ---
def run_kg_extraction_raw():
    if not os.path.exists(INPUT_FILE):
        print(f"Error: {INPUT_FILE} not found.")
        return

    print(f"Starting RAW Extraction. Saving results to: {OUTPUT_FILE}")

    # We use 'w' to overwrite. Switch to 'a' if you need to append to an existing run.
    with open(INPUT_FILE, 'r', encoding='utf-8') as f, \
         open(OUTPUT_FILE, 'w', encoding='utf-8') as out_f:
        
        # Write a file header
        out_f.write("# Monte Cristo Raw Knowledge Graph Extraction\n")
        out_f.write(f"Generated on: {time.strftime('%Y-%m-%d %H:%M:%S')}\n\n")

        for i, line in enumerate(f):
            chunk = json.loads(line)
            summary = chunk.get("causal_summary", "")
            logic_type = chunk.get("logic_type", "N/A")
            
            # API Call
            try:
                response = client.chat.complete(
                    model=MODEL_ID,
                    messages=[
                        {"role": "system", "content": GRAPH_CONSTRUCT_PROMPT},
                        {"role": "user", "content": f"Entity_types: [person, organization, geo, event]\nText: {summary}"}
                    ],
                    temperature=0.1
                )
                raw_data = response.choices[0].message.content
                
                # --- SAVE RAW DATA TO MARKDOWN ---
                out_f.write(f"### Chunk #{i} | Logic: {logic_type}\n")
                out_f.write(f"**Source Summary:** {summary}\n\n")
                out_f.write(f"**Raw Extraction:**\n\n{raw_data}\n\n")
                out_f.write("---\n\n") # Horizontal rule separator
                
                # Immediate flush ensures safety if the session times out
                out_f.flush()
                
                if i % 10 == 0: 
                    print(f"Chunk #{i} Raw Extraction Complete.")
                
            except Exception as e:
                print(f"Error at Chunk {i}: {e}")
            
            time.sleep(0.5) # Respect API limits

    print(f"\nSuccess! Raw extraction is complete and saved in {OUTPUT_FILE}")

run_kg_extraction_raw()

## 5. Character Name Deduplication
First an audit pass (string-similarity grouping) to find alias clusters by hand, then the exhaustive hand-built alias map used to collapse all variants to one canonical name per character.

In [ ]:
import json
import os
from collections import defaultdict
from difflib import SequenceMatcher

# --- CONFIGURATION ---
INPUT_FILE = '/kaggle/input/monte-cristo-skeleton-and-profiles/monte_cristo_character_profiles (1).json'

def get_similarity(a, b):
    """Calculates string similarity ratio."""
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

def inspect_profile_names():
    if not os.path.exists(INPUT_FILE):
        print(f"Error: {INPUT_FILE} not found at {INPUT_FILE}")
        return

    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        profiles = json.load(f)
    
    # Get all names (keys) from the dictionary and clean whitespace
    all_names = sorted([name.strip() for name in profiles.keys()])
    
    print(f"--- COMPLETE NAME INVENTORY ({len(all_names)} total) ---")
    for name in all_names:
        print(f"• {name}")
    
    print("\n" + "="*50 + "\n")

    # Grouping by similarity logic
    groups = defaultdict(list)
    processed = set()

    print(f"--- INSPECTING POTENTIAL ALIAS GROUPS ---\n")

    for name in all_names:
        if name in processed: continue
        
        group_key = name
        groups[group_key].append(name)
        processed.add(name)
        
        for other_name in all_names:
            if other_name in processed: continue
            
            # Threshold set to 0.75 to catch variations like "Edmond Dantes" vs "Edmond Dantès"
            if get_similarity(name, other_name) > 0.75:
                groups[group_key].append(other_name)
                processed.add(other_name)

    # Output results for manual review
    found_dupes = False
    for master, aliases in groups.items():
        if len(aliases) > 1:
            found_dupes = True
            print(f"Group Found: {master}")
            for alias in aliases:
                print(f"  └─ {alias}")
            print()
            
    if not found_dupes:
        print("No high-similarity duplicates were found in the profile names.")

inspect_profile_names()

In [ ]:
import json
import os

# --- 1. CONFIGURATION ---
INPUT_FILE = '/kaggle/input/monte-cristo-skeleton-and-profiles/monte_cristo_character_profiles (1).json'
OUTPUT_DIR = '/kaggle/working/the_count_of_monte_cristo'
OUTPUT_FILE = os.path.join(OUTPUT_DIR, 'monte_cristo_deduped_profiles_final.json')

# --- 2. THE EXHAUSTIVE MASTER ALIAS MAP ---
# Derived from the 210-name inventory to ensure 100% resolution accuracy.
MASTER_MAP = {
    "Edmond Dantès": [
        "Abbé Busoni", "Count Monte Cristo", "Count de Monte Cristo", "Count of Monte Cristo", 
        "Count of Monte Cristo (Edmond Dantès)", "Count of Monte Cristo (M. Zaccone)", 
        "Count of Monte Cristo (formerly Lord Wilmore)", "Edmond Dantès (Count of Monte Cristo)", 
        "Edmond Dantès (Monte Cristo)", "Edmond Dantès (The Count)", "Edmond Dantès (as 'Andrea')", 
        "Edmond Dantès (as Count de Monte Cristo)", "Edmond Dantès (as Maximilian Morrel)", 
        "Edmond Dantès (as Monte Cristo)", "Edmond Dantès (implied narrator)", 
        "Edmond Dantès (implied via Faria's narration)", "Edmond Dantès (implied)", 
        "Edmond Dantès (implied, via context)", "Edmond Dantès (referred to as Morrel’s son, later revealed as the protagonist)", 
        "Edmond Dantès / Count of Monte Cristo", "Lord Wilmore", "Monte Cristo (Edmond Dantès)", 
        "Narrator (Edmond Dantès)", "Narrator (Monk-like figure, implied to be Edmond Dantès from *The Count of Monte Cristo*)", 
        "Narrator (Monk-like figure, likely Edmond Dantès from *The Count of Monte Cristo*)", 
        "Narrator (Monte Cristo or similar, implied by context)", "Narrator (likely Edmond Dantès)", 
        "Sinbad", "Sinbad the Sailor", "The Count", "The Count (Edmond Dantès)", "The Count (Monte Cristo)", 
        "The Count (likely Edmond Dantès)", "The Count of Monte Cristo", "The Narrator (later revealed as Edmond Dantès)", 
        "The Protagonist (Unnamed)", "Unnamed Narrator (Presumed Protagonist)", "Unnamed Protagonist (Speaker)", 
        "Unnamed Protagonist (later revealed as Edmond Dantès)", "Unnamed Protagonist (likely Edmond Dantès, based on context)", 
        "Édmond Dantès", "M. Zaccone", "Number 34", "The Count (Rocca Priori)", "The Count (unspecified identity, implied as a wealthy outsider)",
        "Monte Cristo", "Sinbad", "The Count"
    ],
    "Fernand Mondego": [
        "Fernand", "Comte de Morcerf", "Count Morcerf", "Count de Morcerf", "Count of Morcerf", 
        "Fernand Mondego (Albert’s father)", "Fernand Mondego (Count of Morcerf)", 
        "Fernand Mondego (later Count de Monte Cristo)", "Fernand Mondego (later Count of Monte Cristo)", 
        "Fernand Montresor", "Fernand Morcerf", "Fernand/Morcerf", "M. de Morcerf (Count de Morcerf)"
    ],
    "Gérard de Villefort": [
        "Villefort", "M. de Villefort", "Gérard de Villefort", "Villefort (and his father)", 
        "Fernand Mondego (Villefort)", "Fernand Mondego (M. de Villefort)", "Fernand Mondego (as Villefort)", 
        "Fernand Mondego (referred to as M. de Villefort)"
    ],
    "Baron Danglars": [
        "Danglars", "Baron Danglars", "Baron Franz Danglars", "Baron François Danglars", 
        "Edmond Danglars", "Franz Danglars", "M. Morrel (as Danglars context)", "Monte Cristo (Danglars)"
    ],
    "Benedetto": [
        "Andrea", "Andrea Benedetto", "Andrea Cavalcanti", "Andrea del Bonzo", "Benedetto", 
        "Benedetto (Andrea Cavalcanti)", "Benedetto/Mercédès (Andrea)", "Major Cavalcanti (Andrea Cavalcanti)", 
        "Young Man (later revealed as Count Andrea Cavalcanti)"
    ],
    "Valentine de Villefort": [
        "Valentine", "Valentine de Noirtier", "Valentine de Saint-Méran", "Valentine de Villefort", 
        "Mademoiselle de Villefort", "Marchioness (Valentine’s Grandmother)"
    ],
    "Albert de Morcerf": [
        "Albert", "Albert (Count of Morcerf)", "Albert (Viscount)", "Albert (the Frenchman)", 
        "Albert Morcerf", "Albert de Morcerf", "Albert de Morcerf (Vicomte de Morcerf)"
    ],
    "Mercédès": [
        "Mercédès", "Mercédès de Morcerf", "Madame de Morcerf (Mercédès)", "Benedetto/Mercédès (Andrea)"
    ],
    "Maximilian Morrel": [
        "Maximilian", "Maximilian Morrel", "Maximilian de Morrel", "Edmond Dantès (as Maximilian Morrel)"
    ],
    "Abbé Faria": [
        "Abbé Faria", "Abbé Faria (implied through Monte Cristo)", "Faria", "Faria (the Abbé)", 
        "Unnamed Elderly Stranger (later revealed as possibly Abbe Faria)"
    ],
    "Noirtier de Villefort": [
        "Noirtier", "Noirtier de Villefort", "M. Noirtier", "M. Noirtier de Villefort"
    ],
    "Franz d’Épinay": [
        "Franz", "Franz d'Epinay", "Franz d’Épinay", "Franz de Quesnel, baron d’Épinay", 
        "Franz d’Elsa", "Franz d’Elzia", "Franz d’Erlach", "Albert de Morcerf / Franz d’Épinay"
    ],
    "M. Morrel": [
        "M. Morrel", "Morrel", "Pierre Morrel", "Edmond Dantès (Morrel)"
    ],
    "Madame Danglars": [
        "Madame Danglars", "Madame Danglars (Baroness)", "Baroness", "Baroness (Debray's wife)", "Unnamed Female Character (Baroness)"
    ],
    "Eugénie Danglars": [
        "Eugénie", "Eugénie Danglars", "Mademoiselle Eugénie Danglars"
    ],
    "Giovanni Bertuccio": [
        "Bertuccio", "Giovanni Bertuccio", "M. Bertuccelli"
    ],
    "Luigi Vampa": [
        "Luigi", "Luigi Vampa", "Vampa"
    ],
    "Caderousse": [
        "Caderousse", "Gaspard Caderousse"
    ],
    "Ali Pasha": [
        "Ali Pasha (Haydée's father)", "Vizier Ali", "Ali Tepelini"
    ],
    "Peppino": [
        "Peppino", "Peppino (likely a pseudonym for a French operative)", "Unnamed Stranger (later revealed as Peppino’s associate)"
    ]
}

def get_master_name(raw_name):
    """Resolves a raw name string to its canonical master identity."""
    raw_name_clean = raw_name.strip()
    raw_name_upper = raw_name_clean.upper()
    
    for master, aliases in MASTER_MAP.items():
        # Direct check
        if raw_name_clean == master:
            return master
        # Alias check (case-insensitive)
        if any(alias.upper() == raw_name_upper for alias in aliases):
            return master
        # Handle parenthetical variations found in inventory
        if any(f"({alias.upper()})" in raw_name_upper for alias in aliases):
            return master
            
    return raw_name_clean

def merge_data(target, source):
    """Deeply merges elaborate lists (attributes, locations, timeline) while preserving structure."""
    for key in ['attributes', 'locations', 'logic_timeline']:
        if key in source and isinstance(source[key], list):
            if key not in target:
                target[key] = []
            for item in source[key]:
                # Deep comparison check to prevent duplicate dictionaries or strings
                if item not in target[key]:
                    target[key].append(item)

# --- 3. EXECUTION ENGINE ---
def run_deduplication():
    if not os.path.exists(INPUT_FILE):
        print(f"Error: {INPUT_FILE} not found.")
        return

    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)

    final_profiles = {}

    for raw_name, content in raw_data.items():
        master = get_master_name(raw_name)
        
        if master not in final_profiles:
            final_profiles[master] = {
                "canonical_name": master,
                "original_mentions": [raw_name],
                "attributes": [],
                "locations": [],
                "logic_timeline": []
            }
        else:
            if raw_name not in final_profiles[master]["original_mentions"]:
                final_profiles[master]["original_mentions"].append(raw_name)
        
        # Aggregate the elaborate content arrays
        if isinstance(content, dict):
            merge_data(final_profiles[master], content)

    # Ensure output directory and save
    os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        json.dump(list(final_profiles.values()), f, indent=4, ensure_ascii=False)

    print(f"Success! Deduplicated {len(raw_data)} inventory names into {len(final_profiles)} master entities.")
    print(f"Master file saved: {OUTPUT_FILE}")

run_deduplication()

In [ ]:
import json
import os

# --- CONFIGURATION ---
DEDUPED_FILE = '/kaggle/working/the_count_of_monte_cristo/monte_cristo_deduped_profiles_final.json'

def view_deduped_stats():
    if not os.path.exists(DEDUPED_FILE):
        print(f"Error: {DEDUPED_FILE} not found. Please run the deduplication script first.")
        return

    with open(DEDUPED_FILE, 'r', encoding='utf-8') as f:
        master_profiles = json.load(f)

    # 1. Total Unique Count
    unique_count = len(master_profiles)
    
    print(f"--- DEDUPLICATION SUMMARY ---")
    print(f"Total Unique Master Identities: {unique_count}")
    print("-" * 30 + "\n")

    # 2. Detailed Breakdown (First 20 for brevity)
    print(f"--- MASTER NODE INVENTORY (Top 20) ---")
    for i, profile in enumerate(master_profiles[:20]):
        name = profile.get('canonical_name')
        mention_count = len(profile.get('original_mentions', []))
        attr_count = len(profile.get('attributes', []))
        logic_count = len(profile.get('logic_timeline', []))
        
        print(f"{i+1}. {name}")
        print(f"   └─ Resolved from: {mention_count} different aliases")
        print(f"   └─ Features: {attr_count} attributes, {logic_count} timeline events")
    
    # 3. Highlight the "Protagonist" Node Depth
    edmond = next((p for p in master_profiles if p['canonical_name'] == "Edmond Dantès"), None)
    if edmond:
        print("\n" + "="*50)
        print(f"PROTAGONIST DEPTH CHECK: {edmond['canonical_name']}")
        print(f"Total logic events captured: {len(edmond['logic_timeline'])}")
        print(f"First 2 timeline states:")
        for state in edmond['logic_timeline'][:2]:
            print(f"  - [{state['type']}] {state['summary'][:100]}...")
        print("="*50)

view_deduped_stats()

## 6. Knowledge Graph Assembly
Two passes: a base graph built directly from the deduplicated profiles + extracted triplets, then an improved version with fuzzy/thematic name resolution and evidence injection (the version used downstream).

In [ ]:
import json
import re
import os

# --- 1. PATHS ---
DEDUPED_PROFILES = '/kaggle/working/monte_cristo_deduped_profiles_final.json'
KG_MD_INPUT = '/kaggle/working/Knowledge graph/monte_cristo_light_KG.md'
FINAL_GRAPH_OUTPUT = '/kaggle/working/Knowledge graph/monte_cristo_full_kg_final.json'

# --- 2. LOAD MASTER DATA ---
with open(DEDUPED_PROFILES, 'r', encoding='utf-8') as f:
    master_profiles = json.load(f)

# Create a lookup for FAST resolution: Alias -> Canonical Name
# We use the 'original_mentions' and 'aliases' stored in your deduped file
name_resolver = {}
for profile in master_profiles:
    canonical = profile['canonical_name']
    name_resolver[canonical.upper()] = canonical
    for alias in profile.get('aliases', []):
        name_resolver[alias.upper()] = canonical
    for mention in profile.get('original_mentions', []):
        name_resolver[mention.upper()] = canonical

def resolve(name):
    name_clean = name.strip().strip('"').upper()
    return name_resolver.get(name_clean, name.strip())

# --- 3. DEDUPLICATE MD TRIPLETS ---
def build_final_graph():
    if not os.path.exists(KG_MD_INPUT):
        print("Error: Markdown KG not found.")
        return

    # Initialize graph structure using Master Profiles as starting nodes
    graph = {
        "nodes": {p['canonical_name']: p for p in master_profiles},
        "edges": {} 
    }

    with open(KG_MD_INPUT, 'r', encoding='utf-8') as f:
        content = f.read()

    # Step A: Aggregate NEW descriptions for existing nodes
    entities = re.findall(r'\("entity"<\|>(.*?)<\|>(.*?)<\|>(.*?)\)', content)
    for name, _, desc in entities:
        master_name = resolve(name)
        if master_name in graph["nodes"]:
            if desc.strip() not in graph["nodes"][master_name]["attributes"]:
                graph["nodes"][master_name]["attributes"].append(desc.strip())

    # Step B: Aggregate and Weight Relationships
    rels = re.findall(r'\("relationship"<\|>(.*?)<\|>(.*?)<\|>(.*?)<\|>(.*?)<\|>(.*?)\)', content)
    for src, tgt, desc, keys, strength in rels:
        m_src = resolve(src)
        m_tgt = resolve(tgt)
        
        if m_src == m_tgt: continue # Ignore self-loops
        
        # Create a unique ID for the edge (undirected for simplicity in GNN)
        edge_id = tuple(sorted([m_src, m_tgt]))
        
        if edge_id not in graph["edges"]:
            graph["edges"][edge_id] = {"descriptions": [], "weights": [], "keywords": []}
        
        graph["edges"][edge_id]["descriptions"].append(desc.strip())
        graph["edges"][edge_id]["keywords"].append(keys.strip())
        try:
            # Clean non-numeric strengths like "**strong**"
            clean_strength = int(re.sub(r'\D', '', strength))
            graph["edges"][edge_id]["weights"].append(clean_strength)
        except:
            graph["edges"][edge_id]["weights"].append(5)

    # Step C: Finalize Edge Weights for Adjacency Matrix
    final_edges = []
    for (node1, node2), data in graph["edges"].items():
        avg_weight = sum(data["weights"]) / len(data["weights"])
        final_edges.append({
            "source": node1,
            "target": node2,
            "weight": round(avg_weight, 2),
            "summary_desc": data["descriptions"][0] # Take first or join them
        })

    # Save finalized graph
    output = {"nodes": list(graph["nodes"].values()), "edges": final_edges}
    with open(FINAL_GRAPH_OUTPUT, 'w', encoding='utf-8') as f:
        json.dump(output, f, indent=4, ensure_ascii=False)

    print(f"Final KG built! Nodes: {len(output['nodes'])}, Edges: {len(output['edges'])}")

build_final_graph()

In [ ]:
pip install pyvis

In [ ]:
import json
import os

# --- CONFIGURATION ---
FINAL_GRAPH_FILE = '/kaggle/working/Knowledge graph/monte_cristo_full_kg_final.json'

def inspect_kg_nodes():
    if not os.path.exists(FINAL_GRAPH_FILE):
        print(f"Error: {FINAL_GRAPH_FILE} not found. Please run the build_final_graph script first.")
        return

    with open(FINAL_GRAPH_FILE, 'r', encoding='utf-8') as f:
        graph_data = json.load(f)

    # 1. Collect all Master Names from the 'nodes' list
    # These are the ones derived from your profile JSON
    profile_nodes = sorted([node['canonical_name'] for node in graph_data.get('nodes', [])])

    # 2. Collect all names mentioned in the 'edges' list
    # This identifies "Discovered" nodes that might be duplicates
    edge_names = set()
    for edge in graph_data.get('edges', []):
        edge_names.add(edge['source'])
        edge_names.add(edge['target'])
    
    edge_names = sorted(list(edge_names))

    print(f"--- KNOWLEDGE GRAPH NODE AUDIT ---")
    print(f"Total Nodes defined in Profiles: {len(profile_nodes)}")
    print(f"Total Unique Names found in Relationships: {len(edge_names)}")
    print("-" * 40)

    print("\n[PART A] NODES FROM PROFILE (MASTER LIST):")
    for name in profile_nodes:
        print(f"• {name}")

    print("\n[PART B] NAMES FOUND IN RELATIONSHIPS (The .md Extractions):")
    print("Check if any of these should have been resolved to a name in Part A.")
    for name in edge_names:
        status = "✅ MATCHED" if name in profile_nodes else "❌ UNRESOLVED (New Node)"
        print(f"• {name: <30} | {status}")

    # 3. Identify potential duplicates (names in Part B but not Part A)
    unresolved = [n for n in edge_names if n not in profile_nodes]
    if unresolved:
        print("\n" + "!" * 40)
        print(f"ALERT: {len(unresolved)} names are acting as duplicate nodes!")
        print("These need to be added to your ALIAS_MAP.")
        print("!" * 40)
        for name in unresolved:
            print(f"- {name}")

inspect_kg_nodes()

In [ ]:
import json
import re
import os
from difflib import SequenceMatcher

# --- CONFIGURATION ---
DEDUPED_PROFILES = '/kaggle/working/the_count_of_monte_cristo/monte_cristo_deduped_profiles_final.json'
KG_MD_INPUT = '/kaggle/working/Knowledge graph/monte_cristo_light_KG.md'
FINAL_STORY_KG = '/kaggle/working/Knowledge graph/monte_cristo_story_wide_kg.json'

with open(DEDUPED_PROFILES, 'r', encoding='utf-8') as f:
    master_profiles = json.load(f)

# 1. Expand the lookup to include EVERY alias we've ever seen
name_resolver = {}
for p in master_profiles:
    can = p['canonical_name']
    name_resolver[can.upper()] = can
    for alias in p.get('aliases', []):
        name_resolver[alias.upper()] = can
    for mention in p.get('original_mentions', []):
        name_resolver[mention.upper()] = can

def get_similarity(a, b):
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

def resolve_anything(name):
    """Algorithmic resolution: Profile Match -> Fuzzy Match -> Thematic Group."""
    clean = name.strip().strip('*').strip('"')
    upper = clean.upper()

    # A. Check the Master Map first
    if upper in name_resolver:
        return name_resolver[upper]

    # B. Fuzzy Search: Check if it's 85% similar to a Master Character
    for master_name in name_resolver.values():
        if get_similarity(clean, master_name) > 0.85:
            return master_name

    # C. Thematic Grouping (The "Story Connective Tissue")
    # Instead of hardcoding, we use simple regex 'contains'
    theme_rules = {
        "Wealth/Money": ["FRANC", "LIVRE", "GOLD", "TREASURE", "WEALTH", "FRCS"],
        "Event: Bonapartist Plot": ["BONAPART", "NAPOLEON", "USURPER", "ELBA"],
        "Concept: Vengeance": ["VENGE", "RETRIBUTION", "PUNISH"],
        "Concept: Emotion/Trauma": ["GRIEF", "DESPAIR", "SUFFER", "GUILT"],
        "Location: Chateau d'If": ["CHATEAU", "CHÂTEAU", "D'IF", "DUNGEON", "PRISON"]
    }

    for theme, keywords in theme_rules.items():
        if any(kw in upper for kw in keywords):
            return theme

    # D. Default: Keep it as a unique Story Element if it's novel
    return clean

# --- 2. THE REBUILDER ENGINE ---
def build_story_kg():
    nodes = {p['canonical_name']: {"type": "Character", "data": p} for p in master_profiles}
    edges = {}

    with open(KG_MD_INPUT, 'r', encoding='utf-8') as f:
        content = f.read()

    rels = re.findall(r'\("relationship"<\|>(.*?)<\|>(.*?)<\|>(.*?)<\|>(.*?)<\|>(.*?)\)', content)
    
    for src, tgt, desc, keys, strength in rels:
        m_src = resolve_anything(src)
        m_tgt = resolve_anything(tgt)
        
        if m_src == m_tgt: continue
        
        # Ensure all resolved entities have a node entry
        for n in [m_src, m_tgt]:
            if n not in nodes:
                nodes[n] = {"type": "Story Element", "attributes": []}

        edge_id = tuple(sorted([m_src, m_tgt]))
        if edge_id not in edges:
            edges[edge_id] = {"weights": [], "descriptions": []}
        
        try:
            val = int(re.sub(r'\D', '', strength))
            edges[edge_id]["weights"].append(val)
        except:
            edges[edge_id]["weights"].append(5)
        edges[edge_id]["descriptions"].append(desc)

    # Re-structure for JSON
    final_nodes = [{"id": k, **v} for k, v in nodes.items()]
    final_edges = []
    for (n1, n2), data in edges.items():
        final_edges.append({
            "source": n1,
            "target": n2,
            "weight": round(sum(data["weights"]) / len(data["weights"]), 2),
            "description": data["descriptions"][0]
        })

    with open(FINAL_STORY_KG, 'w', encoding='utf-8') as f:
        json.dump({"nodes": final_nodes, "edges": final_edges}, f, indent=4)

    print(f"Refinery Complete: {len(final_nodes)} unique nodes after algorithmic deduplication.")

build_story_kg()

In [ ]:
import json
import re
import os

# --- PATHS ---
DEDUPED_PROFILES = '/kaggle/working/the_count_of_monte_cristo/monte_cristo_deduped_profiles_final.json'
KG_MD_INPUT = '/kaggle/working/Knowledge graph/monte_cristo_light_KG.md'
ENRICHED_PROFILES = '/kaggle/working/monte_cristo_enriched_profiles.json'

with open(DEDUPED_PROFILES, 'r', encoding='utf-8') as f:
    profiles = json.load(f)

# Create a lookup for all aliases to their master profile
resolver = {}
for p in profiles:
    name = p['canonical_name']
    resolver[name.upper()] = p
    for alias in p.get('aliases', []):
        resolver[alias.upper()] = p
    for mention in p.get('original_mentions', []):
        resolver[mention.upper()] = p

# --- 2. THE INJECTION ENGINE ---
def enrich_profiles():
    with open(KG_MD_INPUT, 'r', encoding='utf-8') as f:
        md_content = f.read()

    # Find all entity descriptions in the .md
    # Format: ("entity"<|><name><|><type><|><desc>)
    entity_extractions = re.findall(r'\("entity"<\|>(.*?)<\|>(.*?)<\|>(.*?)\)', md_content)
    
    for name, _, desc in entity_extractions:
        master_profile = resolver.get(name.strip().upper())
        if master_profile:
            # Inject this extracted description as a new attribute
            if "extracted_evidence" not in master_profile:
                master_profile["extracted_evidence"] = []
            if desc.strip() not in master_profile["extracted_evidence"]:
                master_profile["extracted_evidence"].append(desc.strip())

    # Save the new, data-rich profiles
    with open(ENRICHED_PROFILES, 'w', encoding='utf-8') as f:
        json.dump(profiles, f, indent=4, ensure_ascii=False)

    print(f"Enrichment Complete. Your profiles now contain raw evidence from the 844+ chunks.")

enrich_profiles()

In [ ]:
import json
import re
import os
import numpy as np

# --- 1. CONFIGURATION ---
ENRICHED_PROFILES = '/kaggle/working/monte_cristo_enriched_profiles.json'
KG_MD_INPUT = '/kaggle/working/Knowledge graph/monte_cristo_light_KG.md'
FINAL_GNN_READY_JSON = '/kaggle/working/Knowledge graph/monte_cristo_gnn_ready.json'

def build_evidence_synthesized_graph():
    # Load the profiles that now contain injected evidence
    with open(ENRICHED_PROFILES, 'r', encoding='utf-8') as f:
        profiles = json.load(f)

    # A. INITIALIZE NODES (The Social Map)
    # We map every alias to a canonical index for the Adjacency Matrix
    master_nodes = {}
    resolver = {}
    
    for i, p in enumerate(profiles):
        name = p['canonical_name']
        master_nodes[name] = {
            "index": i,
            "profile_data": p,
            "type": "Person"
        }
        # Build the resolution lookup for the .md triplets
        resolver[name.upper()] = name
        for alias in p.get('aliases', []):
            resolver[alias.upper()] = name
        for mention in p.get('original_mentions', []):
            resolver[mention.upper()] = name

    # B. SYNTHESIZE EDGES (Relationship Evidence)
    edges = {}
    
    with open(KG_MD_INPUT, 'r', encoding='utf-8') as f:
        content = f.read()

    # Extract all triplets from the .md
    rels = re.findall(r'\("relationship"<\|>(.*?)<\|>(.*?)<\|>(.*?)<\|>(.*?)<\|>(.*?)\)', content)
    
    for src, tgt, desc, keys, strength in rels:
        m_src = resolver.get(src.strip().upper())
        m_tgt = resolver.get(tgt.strip().upper())
        
        # VALIDATION: Only record relationships between confirmed Master Entities
        if m_src and m_tgt and m_src != m_tgt:
            edge_id = tuple(sorted([m_src, m_tgt]))
            
            if edge_id not in edges:
                edges[edge_id] = {
                    "weight_accumulator": [],
                    "evidence_summaries": [],
                    "narrative_keys": set()
                }
            
            # Aggregate the evidence
            edges[edge_id]["evidence_summaries"].append(desc.strip())
            edges[edge_id]["narrative_keys"].update(keys.split(','))
            try:
                # Clean strength from markdown formatting (e.g., "**8**")
                s = int(re.sub(r'\D', '', strength))
                edges[edge_id]["weight_accumulator"].append(s)
            except:
                edges[edge_id]["weight_accumulator"].append(5)

    # C. FINALIZE STRUCTURE
    final_edges = []
    for (n1, n2), data in edges.items():
        avg_weight = sum(data["weight_accumulator"]) / len(data["weight_accumulator"])
        final_edges.append({
            "source": n1,
            "target": n2,
            "source_idx": master_nodes[n1]["index"],
            "target_idx": master_nodes[n2]["index"],
            "weight": round(avg_weight, 2),
            "evidence": data["evidence_summaries"][:5], # Keep top 5 samples
            "keywords": list(data["narrative_keys"])
        })

    # Prepare final output
    output_kg = {
        "metadata": {
            "node_count": len(master_nodes),
            "edge_count": len(final_edges),
            "source_chunks": 844
        },
        "nodes": list(master_nodes.values()),
        "edges": final_edges
    }

    with open(FINAL_GNN_READY_JSON, 'w', encoding='utf-8') as f:
        json.dump(output_kg, f, indent=4, ensure_ascii=False)

    print(f"KG Synthesis Complete!")
    print(f"Matrix Size: {len(master_nodes)}x{len(master_nodes)}")
    print(f"Total Relational Evidence Paths: {len(final_edges)}")

build_evidence_synthesized_graph()

In [ ]:
import json
import os
from pyvis.network import Network

# --- 1. CONFIGURATION ---
INPUT_GRAPH = '/kaggle/working/Knowledge graph/monte_cristo_gnn_ready.json'
OUTPUT_HTML = '/kaggle/working/Knowledge graph/monte_cristo_synthesized_kg.html'

def visualize_synthesized_kg():
    if not os.path.exists(INPUT_GRAPH):
        print(f"Error: {INPUT_GRAPH} not found.")
        return

    # Load the character-only synthesized graph
    with open(INPUT_GRAPH, 'r', encoding='utf-8') as f:
        graph_data = json.load(f)

    # Initialize Pyvis Network
    net = Network(height='800px', width='100%', bgcolor='#0f0f0f', font_color='white', notebook=True, directed=False)
    
    # Use ForceAtlas2 for better spatial clustering of social blocks
    net.force_atlas_2based()

    # --- 2. ADD CHARACTER NODES ---
    for node in graph_data['nodes']:
        name = node['index'] # Use index for ID
        label = node['profile_data']['canonical_name']
        
        # Scale node size by the amount of evidence we've injected
        evidence_count = len(node['profile_data'].get('extracted_evidence', []))
        size = 15 + (evidence_count * 0.5) 
        
        # Color coding: Deep teal for primary characters
        color = '#008080' if evidence_count > 10 else '#4682B4'
        
        # Build hover title from the injected evidence
        evidence_list = node['profile_data'].get('extracted_evidence', [])[:5]
        title = f"<b>{label}</b><br><i>Evidence Points: {evidence_count}</i><br><br>"
        title += "<br>".join([f"• {e[:80]}..." for e in evidence_list])
        
        net.add_node(label, label=label, title=title, color=color, size=size)

    # --- 3. ADD SYNTHESIZED EDGES ---
    for edge in graph_data['edges']:
        src = edge['source']
        tgt = edge['target']
        weight = edge['weight']
        
        # Hover title for edges shows the relationship keywords
        keywords = ", ".join(edge.get('keywords', []))
        title = f"<b>Relationship:</b> {edge['source']} & {edge['target']}<br><b>Strength:</b> {weight}<br><b>Keys:</b> {keywords}"
        
        # Value controls thickness; title controls hover text
        net.add_edge(src, tgt, value=weight, title=title, color='#708090', alpha=0.6)

    # --- 4. GENERATE AND SAVE ---
    net.show(OUTPUT_HTML)
    print(f"Synthesized KG visualization created: {OUTPUT_HTML}")
    print(f"Visualizing {len(graph_data['nodes'])} characters and their evidenced links.")

visualize_synthesized_kg()

## 7. Final Entity Resolution & Deep Merge
A final audit of canonical names across graph files, a consolidated alias map, and a lossless deep-merge that aggregates every field across all node variants into one hub profile per character.

In [ ]:
import json

files = ['/kaggle/working/Knowledge graph/monte_cristo_gnn_ready.json']

def extract_canonical_names(file_list):
    all_names = {}
    
    for file_path in file_list:
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                nodes = data.get('nodes', [])
                
                for node in nodes:
                    # In full_kg_final, it's a top-level key. 
                    # In gnn_ready, it's inside profile_data.
                    name = node.get('canonical_name') or \
                           node.get('profile_data', {}).get('canonical_name')
                    
                    if name:
                        if name not in all_names:
                            all_names[name] = []
                        all_names[name].append(file_path)
        except FileNotFoundError:
            print(f"File not found: {file_path}")
            
    return all_names

# Execute and Print
names_dict = extract_canonical_names(files)

print(f"{'Canonical Name':<50} | {'Found In'}")
print("-" * 75)
for name in sorted(names_dict.keys()):
    found_in = ", ".join(set(names_dict[name]))
    print(f"{name:<50} | {found_in}")

print(f"\nTotal Unique Canonical Names found: {len(names_dict)}")

In [ ]:
alias_map = {
    # Edmond Dantès Hub
    "Edmond Dantès (implied via context, though not explicitly named here)": "Edmond Dantès",
    "Sinbad the Sailor": "Edmond Dantès",
    "The Count of Monte Cristo": "Edmond Dantès",
    "Lord Wilmore": "Edmond Dantès",
    "Abbé Busoni": "Edmond Dantès",
    
    # The Villefort/Morcerf Clusters
    "Edward": "Edward de Villefort",
    "Bernard (Villefort's father)": "Noirtier de Villefort",
    "Albert Danglars": "Albert de Morcerf",
    "Albert (and Franz, by extension)": "Albert de Morcerf",
    "Fernand Danglars": "Fernand Mondego",
    
    # The d'Épinay/Quesnel Cluster (Critical for the Duel logic)
    "General Flavien de Quesnel": "General Flavien de Quesnel",
    "General de Quesnel": "General Flavien de Quesnel",
    "General d’Épinay": "General Flavien de Quesnel",
    "M. d’Épinay (General)": "General Flavien de Quesnel",
    
    # Family & Secondary Hubs
    "Julie": "Julie Morrel",
    "Louis XVIII": "King Louis XVIII",
    "Marchioness de Saint-Méran": "Madame de Saint-Méran",
    "Major Cavalcanti (Andrea’s father)": "Major Cavalcanti",
    "The Major": "Major Cavalcanti",
    "Benedetto (likely alias of the narrator)": "Benedetto",
    
    # The Eugénie Cluster (Cleaning LLM hallucinations)
    "Eugène Danglars": "Eugénie Danglars",
    "Eugénie de Rênal": "Eugénie Danglars",
    "Eugénie d’Armilly": "Eugénie Danglars",
    
    # Role-based Hubs
    "Gardener (Telegraph Operator)": "Telegraph Operator",
    "Telegraph Operator (Unnamed)": "Telegraph Operator",
    "The Telegraph Operator (Unnamed Man)": "Telegraph Operator",
    "Unnamed Gardener": "Telegraph Operator"
}

In [ ]:
import json

# Define the absolute path
file_path = '/kaggle/working/Knowledge graph/monte_cristo_gnn_ready.json'

def print_json_structure(path):
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    print("=== ROOT LEVEL KEYS ===")
    for key in data.keys():
        print(f"- {key} ({type(data[key]).__name__})")

    if 'nodes' in data and len(data['nodes']) > 0:
        print("\n=== NODE STRUCTURE (profile_data) ===")
        # Get all unique keys found across all nodes' profile_data
        node_keys = {}
        for node in data['nodes']:
            profile = node.get('profile_data', {})
            for pk, pv in profile.items():
                if pk not in node_keys:
                    node_keys[pk] = type(pv).__name__
        
        for k, t in node_keys.items():
            print(f"- {k}: {t}")

    if 'edges' in data and len(data['edges']) > 0:
        print("\n=== EDGE STRUCTURE ===")
        edge_keys = {}
        for edge in data['edges']:
            for ek, ev in edge.items():
                if ek not in edge_keys:
                    edge_keys[ek] = type(ev).__name__
        
        for k, t in edge_keys.items():
            print(f"- {k}: {t}")

# Run the inspector
print_json_structure(file_path)

In [ ]:
import json
import os
from collections import defaultdict

# Update these paths to match your current local environment
input_path = '/kaggle/working/Knowledge graph/monte_cristo_gnn_ready.json'
output_path = '/kaggle/working/clean/monte_cristo_merged_final.json'

def perform_deep_lossless_merge(input_path, alias_map, output_path):
    # Ensure the directory exists
    if os.path.dirname(output_path):
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    # 1. READ with UTF-8
    with open(input_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    old_nodes = data['nodes']
    old_edges = data['edges']
    
    hub_registry = {} 
    old_to_new_idx = {}

    # 2. DYNAMIC NODE MERGE
    for node in old_nodes:
        profile = node.get('profile_data', {})
        raw_name = profile.get('canonical_name', 'Unknown')
        hub_name = alias_map.get(raw_name, raw_name)
        
        if hub_name not in hub_registry:
            hub_registry[hub_name] = {"canonical_name": hub_name, "_old_indices": []}
        
        hub_profile = hub_registry[hub_name]
        hub_profile["_old_indices"].append(node['index'])
        
        for key, value in profile.items():
            if key == "canonical_name": continue
            if key not in hub_profile:
                if isinstance(value, list): hub_profile[key] = []
                elif isinstance(value, dict): hub_profile[key] = []
                else: hub_profile[key] = set()
            
            if isinstance(value, list):
                for item in value:
                    if item not in hub_profile[key]: hub_profile[key].append(item)
            elif isinstance(value, (str, int, float, bool)) or value is None:
                if isinstance(hub_profile[key], set): hub_profile[key].add(value)
                else:
                    if value not in hub_profile[key]: hub_profile[key].append(value)
            elif isinstance(value, dict):
                if value not in hub_profile[key]: hub_profile[key].append(value)

    # 3. RE-INDEX and FINALIZE
    new_nodes = []
    for i, (name, profile) in enumerate(sorted(hub_registry.items())):
        old_indices = profile.pop("_old_indices")
        # Finalizing profile data while preserving characters
        final_profile = {k: (sorted(list(v)) if isinstance(v, set) else v) for k, v in profile.items()}
        new_nodes.append({"index": i, "profile_data": final_profile})
        for old_idx in old_indices:
            old_to_new_idx[old_idx] = i

    # 4. EDGE MERGE: Re-map source/target and aggregate evidence
    merged_edges_map = {}
    for edge in old_edges:
        src_new = old_to_new_idx.get(edge['source_idx'])
        tgt_new = old_to_new_idx.get(edge['target_idx'])
        if src_new is None or tgt_new is None or src_new == tgt_new:
            continue
            
        edge_key = tuple(sorted((src_new, tgt_new)))
        if edge_key not in merged_edges_map:
            merged_edges_map[edge_key] = {
                "source": new_nodes[src_new]["profile_data"]["canonical_name"],
                "target": new_nodes[tgt_new]["profile_data"]["canonical_name"],
                "source_idx": src_new,
                "target_idx": tgt_new,
                "weights": [],
                "evidence": set(),
                "keywords": set()
            }
        
        e_hub = merged_edges_map[edge_key]
        e_hub["weights"].append(edge.get('weight', 0))
        if 'evidence' in edge: e_hub["evidence"].update(edge['evidence'])
        if 'keywords' in edge: e_hub["keywords"].update(edge['keywords'])

    # 5. FINALIZE EDGES
    final_edges = []
    for e in merged_edges_map.values():
        e["weight"] = sum(e["weights"]) / len(e["weights"])
        del e["weights"]
        e["evidence"] = sorted(list(e["evidence"]))
        e["keywords"] = sorted(list(e["keywords"]))
        final_edges.append(e)

    # 6. WRITE with UTF-8 and ensure_ascii=False
    output_data = {
        "metadata": {
            "node_count": len(new_nodes),
            "edge_count": len(final_edges),
            "original_nodes": len(old_nodes)
        },
        "nodes": new_nodes,
        "edges": final_edges
    }
    
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(output_data, f, indent=4, ensure_ascii=False)
    
    print(f"Merge Complete! File saved to: {output_path}")

# Execute using your alias_map
perform_deep_lossless_merge(input_path, alias_map, output_path)

## 8. Hierarchical Attribute Agents
Three stateful LLM agents that deduplicate and normalize a character's locations, attributes/traits, and causal timeline into compact, high-density summaries, then fuse the results into one "star graph" per character.

In [ ]:
import json
import time
import os
from mistralai import Mistral
from kaggle_secrets import UserSecretsClient

# --- 1. CONFIGURATION ---
user_secrets = UserSecretsClient() 
MISTRAL_API_KEY = user_secrets.get_secret("MISTRAL_API_KEY") 
client = Mistral(api_key=MISTRAL_API_KEY)

MODEL_ID = "ministral-8b-latest"
INPUT_PATH = "/kaggle/working/Knowledge graph/monte_cristo_merged_final.json"
OUTPUT_PATH = "/kaggle/working/monte_cristo_spatial_hierarchy.json"

# --- 2. THE STATEFUL AGENT PROMPT ---
SYSTEM_PROMPT = """You are a narrative geography expert. Your goal is to deduplicate locations for a specific character in 'The Count of Monte Cristo'.

### LOGIC:
1. Compare 'NEW CANDIDATES' against 'CURRENT UNIQUE LOCATIONS'.
2. Discard any candidate that is a synonym (e.g., 'The prison' vs 'Château d'If'), a sub-location, or already exists in the unique list.
3. Normalize names to their most recognizable canonical form.

### OUTPUT FORMAT:
You must return ONLY a valid JSON list of strings representing the NEW unique locations found.
Example Output:
["Marseille", "Château d'If", "Island of Monte Cristo"]

If no new locations are found, return:
[]"""

# --- 3. THE AGENT CALL ---
def call_spatial_agent(char_name, current_uniques, batch_candidates):
    # This f-string is the input that is updated after every batch
    input_prompt = f"""
    CHARACTER: {char_name}
    CURRENT UNIQUE LOCATIONS (MEMORY): {current_uniques}
    NEW CANDIDATES TO EVALUATE: {batch_candidates}
    """

    print(f"\n>>> API CALL FOR: {char_name}")
    print(f"    Memory: {current_uniques}")
    print(f"    Batch: {batch_candidates}")

    try:
        response = client.chat.complete(
            model=MODEL_ID,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": input_prompt}
            ],
            response_format={"type": "json_object"},
            temperature=0.1
        )
        
        # Extract and Parse Output
        raw_output = response.choices[0].message.content
        output_json = json.loads(raw_output)
        
        # Ensure we get a list back
        new_items = []
        if isinstance(output_json, list):
            new_items = output_json
        elif isinstance(output_json, dict):
            # Fallback if model wraps list in a key
            for val in output_json.values():
                if isinstance(val, list):
                    new_items = val
                    break
        
        print(f"    >>> AGENT DISCOVERED: {new_items}")
        return new_items

    except Exception as e:
        print(f"    [!] Error: {e}")
        return []

# --- 4. THE SEQUENTIAL EXECUTION ENGINE ---
def run_spatial_deduplication():
    with open(INPUT_PATH, 'r', encoding='utf-8') as f:
        kg = json.load(f)

    spatial_hierarchy = {}

    for node in kg['nodes']:
        # RESET state for new character
        char_name = node['profile_data']['canonical_name']
        raw_locs = node['profile_data'].get('locations', [])
        
        if not raw_locs:
            continue

        print(f"\n" + "="*50)
        print(f"CHARACTER HUB: {char_name}")
        print("="*50)
        
        # State Initialization
        unique_list = [] 

        # Batch Processing (10 at a time)
        for i in range(0, len(raw_locs), 10):
            batch = raw_locs[i : i+10]
            
            # The Synaptic Update: Call agent with growing unique_list
            new_uniques = call_spatial_agent(char_name, unique_list, batch)
            
            # Append new findings to the state memory
            for loc in new_uniques:
                if loc not in unique_list:
                    unique_list.append(loc)
            
            # Slight delay for API stability
            time.sleep(1) 

        spatial_hierarchy[char_name] = unique_list

    # --- 5. SAVE FINAL HIERARCHY ---
    with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
        json.dump(spatial_hierarchy, f, indent=4, ensure_ascii=False)
    
    print(f"\nExtraction Finished. Hierarchy saved to: {OUTPUT_PATH}")

# RUN
run_spatial_deduplication()

In [ ]:
import json
import time
import os
from mistralai import Mistral
from kaggle_secrets import UserSecretsClient

# --- 1. CONFIGURATION ---
user_secrets = UserSecretsClient() 
MISTRAL_API_KEY = user_secrets.get_secret("MISTRAL_API_KEY") 
client = Mistral(api_key=MISTRAL_API_KEY)

MODEL_ID = "ministral-8b-latest"
INPUT_PATH = "/kaggle/working/Knowledge graph/monte_cristo_merged_final.json"
OUTPUT_PATH = "/kaggle/working/monte_cristo_attribute_hierarchy.json"

# --- 2. THE STATEFUL ATTRIBUTE AGENT PROMPT ---
SYSTEM_PROMPT = """You are a character psychology expert for the novel 'The Count of Monte Cristo'.
Your task is to deduplicate and normalize character attributes/traits.

### LOGIC:
1. Compare 'NEW CANDIDATES' against 'CURRENT UNIQUE ATTRIBUTES'.
2. Discard any candidate that is a synonym (e.g., 'Wealthy' vs 'Rich'), a minor variation, or already exists in the unique list.
3. Normalize traits to clear, descriptive terms (e.g., 'Seeking vengeance' -> 'Vengeful').
4. Focus on persistent traits, social status, and psychological states.

### OUTPUT FORMAT:
You must return ONLY a valid JSON list of strings representing the NEW unique attributes found.
Example Output:
["Vengeful", "Wealthy", "Stoic", "Sailor"]

If no new attributes are found, return:
[]"""

# --- 3. THE AGENT CALL ---
def call_attribute_agent(char_name, current_uniques, batch_candidates):
    # Stateful f-string updated every iteration
    input_prompt = f"""
    CHARACTER: {char_name}
    CURRENT UNIQUE ATTRIBUTES (MEMORY): {current_uniques}
    NEW CANDIDATES TO EVALUATE: {batch_candidates}
    """

    print(f"\n>>> API CALL FOR: {char_name}")
    print(f"    Memory: {current_uniques}")
    print(f"    Batch: {batch_candidates}")

    try:
        response = client.chat.complete(
            model=MODEL_ID,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": input_prompt}
            ],
            response_format={"type": "json_object"},
            temperature=0.1
        )
        
        # Extract and Parse
        raw_output = response.choices[0].message.content
        output_json = json.loads(raw_output)
        
        # Ensure list output
        new_items = []
        if isinstance(output_json, list):
            new_items = output_json
        elif isinstance(output_json, dict):
            for val in output_json.values():
                if isinstance(val, list):
                    new_items = val
                    break
        
        print(f"    >>> AGENT DISCOVERED: {new_items}")
        return new_items

    except Exception as e:
        print(f"    [!] Error: {e}")
        return []

# --- 4. THE SEQUENTIAL EXECUTION ENGINE ---
def run_attribute_deduplication():
    with open(INPUT_PATH, 'r', encoding='utf-8') as f:
        kg = json.load(f)

    attribute_hierarchy = {}

    for node in kg['nodes']:
        # RESET state for new character
        char_name = node['profile_data']['canonical_name']
        raw_attrs = node['profile_data'].get('attributes', [])
        
        if not raw_attrs:
            continue

        print(f"\n" + "="*50)
        print(f"CHARACTER HUB: {char_name}")
        print("="*50)
        
        # State Initialization for this specific character
        unique_list = [] 

        # Batch Processing (10 attributes at a time)
        for i in range(0, len(raw_attrs), 10):
            batch = raw_attrs[i : i+10]
            
            # Synaptic Update: Call agent with the growing memory
            new_uniques = call_attribute_agent(char_name, unique_list, batch)
            
            # Append new traits to the character's memory state
            for attr in new_uniques:
                if attr not in unique_list:
                    unique_list.append(attr)
            
            # Delay for API stability
            time.sleep(1) 

        attribute_hierarchy[char_name] = unique_list

    # --- 5. SAVE FINAL HIERARCHY ---
    with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
        json.dump(attribute_hierarchy, f, indent=4, ensure_ascii=False)
    
    print(f"\nAttribute Extraction Finished. Hierarchy saved to: {OUTPUT_PATH}")

# RUN
run_attribute_deduplication()

In [ ]:
import json
import time
import os
from mistralai import Mistral
from kaggle_secrets import UserSecretsClient

# --- 1. CONFIGURATION ---
user_secrets = UserSecretsClient() 
MISTRAL_API_KEY = user_secrets.get_secret("MISTRAL_API_KEY") 
client = Mistral(api_key=MISTRAL_API_KEY)

MODEL_ID = "ministral-8b-latest"
INPUT_PATH = "/kaggle/working/Knowledge graph/monte_cristo_merged_final.json"
OUTPUT_PATH = "/kaggle/working/causal_link/monte_cristo_causal_timeline1.json"

# --- 2. THE MASTER CAUSAL PROMPT ---
# (Kept exactly as requested)
SYSTEM_PROMPT = """You are a Narrative Causal Architect for the novel 'The Count of Monte Cristo'. 
Your task is to consolidate character timeline fragments into high-density Macro-Event Nodes.

### LOGIC CONSTRAINTS:

1. FUSION (CONTINUITY):
   - CRITERIA: Trigger if the underlying text indicates the SAME MOTIVE, criteria, and action as the current event.
   - ACTION: Rewrite the 'Cumulative Summary' into a richer, more descriptive narrative that absorbs the new information.

2. FISSION (STATE JUMP):
   - CRITERIA: Trigger if there is a shift in:
     * SPATIAL: Change in location.
     * CAUSAL: A new cause-effect sequence begins.
     * STATUS: Change in social, physical, or legal state.
     * CLOSURE: The previous conflict or action reached a conclusion.
     * ACTION: A significant, distinct new action is initiated.
   - ACTION: Close the current node and start a new Macro-Event node.

### OUTPUT FORMAT:
Return ONLY a valid JSON object:
{{
  "decision": "FUSION" or "FISSION",
  "fission_reason": "Spatial/Causal/Status/Closure/Action/None",
  "updated_event_name": "3-5 word canonical title",
  "integrated_summary": "The enriched narrative summary (if FUSION) or the starting summary (if FISSION)"
}}"""

# --- 3. THE AGENT CALL ---
# (Kept exactly as requested)
def call_causal_agent(char_name, active_node, candidate_chunk):
    input_prompt = f"""
    ### CONTEXT
    Character: {char_name}

    ### ACTIVE MACRO-EVENT (CURRENT STATE)
    - Name: {active_node['event_name']}
    - Cumulative Summary: {active_node['summary']}

    ### NEW CANDIDATE DETAIL
    - Raw Summary: {candidate_chunk.get('causal_summary', '')}
    - Evidence: {candidate_chunk.get('verbatim_anchor', '')}
    - Logic Type: {candidate_chunk.get('logic_type', 'N/A')}
    """

    print(f"\n>>> ANALYST: {char_name} | Hub: {active_node['event_name']}")

    try:
        response = client.chat.complete(
            model=MODEL_ID,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": input_prompt}
            ],
            response_format={"type": "json_object"},
            temperature=0.1
        )
        
        res = json.loads(response.choices[0].message.content)
        print(f"    DECISION: {res['decision']} ({res.get('fission_reason')}) -> {res['updated_event_name']}")
        return res
    except Exception as e:
        print(f"    [!] API Error: {e}")
        return None

# --- 4. THE RECURSIVE ENGINE (FIXED FOR SAFETY) ---
def run_causal_consolidation():
    # FIX: Ensure directory exists immediately
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

    with open(INPUT_PATH, 'r', encoding='utf-8') as f:
        kg = json.load(f)

    # Resume Logic: Check if some work was already saved
    if os.path.exists(OUTPUT_PATH):
        with open(OUTPUT_PATH, 'r', encoding='utf-8') as f:
            final_timelines = json.load(f)
    else:
        final_timelines = {}

    for node in kg['nodes']:
        char_name = node['profile_data']['canonical_name']
        
        # Skip characters already processed in a previous interrupted run
        if char_name in final_timelines:
            print(f"Skipping {char_name} (Already in save file).")
            continue
            
        raw_timeline = node['profile_data'].get('logic_timeline', [])
        if not raw_timeline: continue

        print(f"\n" + "="*70)
        print(f"CONSOLIDATING CAUSAL SKELETON: {char_name}")
        print("="*70)

        consolidated_list = []
        
        # Initialize the first macro-node
        first = raw_timeline[0]
        current_macro = {
            "event_name": "Initial State",
            "summary": first.get('causal_summary', ''),
            "evidence_stack": [first.get('verbatim_anchor', '')]
        }

        # Iteratively process the timeline
        for i in range(1, len(raw_timeline)):
            candidate = raw_timeline[i]
            res = call_causal_agent(char_name, current_macro, candidate)
            
            if not res: continue

            if res['decision'] == "FUSION":
                current_macro['event_name'] = res['updated_event_name']
                current_macro['summary'] = res['integrated_summary']
                current_macro['evidence_stack'].append(candidate.get('verbatim_anchor', ''))
            else:
                consolidated_list.append(current_macro)
                current_macro = {
                    "event_name": res['updated_event_name'],
                    "summary": res['integrated_summary'],
                    "evidence_stack": [candidate.get('verbatim_anchor', '')]
                }
            
            time.sleep(0.4) 

        # Save the character's final node
        consolidated_list.append(current_macro)
        
        # FIX: Save character results and WRITE to disk immediately (Checkpoint)
        final_timelines[char_name] = consolidated_list
        with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
            json.dump(final_timelines, f, indent=4, ensure_ascii=False)
        print(f"[CHECKPOINT] Saved progress for {char_name}")

    print(f"\nCausal Architecture Complete. Data safely stored in: {OUTPUT_PATH}")

# RUN
run_causal_consolidation()

In [ ]:
import json
import os

# --- 1. FILE CONFIGURATION ---
SPATIAL_PATH = "/kaggle/working/monte_cristo_spatial_hierarchy.json"
ATTRIBUTE_PATH = "/kaggle/working/monte_cristo_attribute_hierarchy.json"
CAUSAL_PATH = "/kaggle/working/causal_link/monte_cristo_causal_timeline1.json"
FINAL_KG_PATH = "/kaggle/working/edmond_dantes_star_graph.json"

def fuse_character_graph(target_char="Edmond Dantès"):
    # Initialize the Star Graph structure
    star_graph = {
        "character_hub": target_char,
        "satellites": {
            "spatial_pillar": [],
            "attribute_pillar": [],
            "causal_pillar": []
        }
    }

    # 2. LOAD & FUSE SPATIAL DATA
    if os.path.exists(SPATIAL_PATH):
        with open(SPATIAL_PATH, 'r') as f:
            spatial_data = json.load(f)
            # Find entry for our target character
            star_graph["satellites"]["spatial_pillar"] = spatial_data.get(target_char, [])

    # 3. LOAD & FUSE ATTRIBUTE DATA
    if os.path.exists(ATTRIBUTE_PATH):
        with open(ATTRIBUTE_PATH, 'r') as f:
            attr_data = json.load(f)
            star_graph["satellites"]["attribute_pillar"] = attr_data.get(target_char, [])

    # 4. LOAD & FUSE CAUSAL TIMELINE (The Dense Nodes)
    if os.path.exists(CAUSAL_PATH):
        with open(CAUSAL_PATH, 'r') as f:
            causal_data = json.load(f)
            # This contains the Macro-Events with integrated summaries and evidence
            star_graph["satellites"]["causal_pillar"] = causal_data.get(target_char, [])

    # --- 5. SAVE FINAL STAR GRAPH ---
    os.makedirs(os.path.dirname(FINAL_KG_PATH), exist_ok=True)
    with open(FINAL_KG_PATH, 'w', encoding='utf-8') as f:
        json.dump(star_graph, f, indent=4, ensure_ascii=False)
    
    print(f"Fusion Complete: {target_char} Star Graph generated at {FINAL_KG_PATH}")

# RUN FUSION
fuse_character_graph("Edmond Dantès")

## 9. Final Visualization

In [ ]:
import json
import os
from pyvis.network import Network

# --- 1. CONFIGURATION ---
INPUT_KG = "/kaggle/working/edmond_dantes_star_graph.json"
OUTPUT_DIR = "/kaggle/working/KG_viz"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "edmond_dantes_viz.html")

# Ensure the output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

def visualize_star_graph():
    # Load the Fused Star Graph
    if not os.path.exists(INPUT_KG):
        print(f"Error: {INPUT_KG} not found. Please run the fusion script first.")
        return

    with open(INPUT_KG, 'r', encoding='utf-8') as f:
        kg = json.load(f)

    # Initialize Pyvis Network
    # height/width for Kaggle notebook display; directed=True for causal flow
    net = Network(height="750px", width="100%", bgcolor="#222222", font_color="white", notebook=False)

    hub_name = kg["character_hub"]
    satellites = kg["satellites"]

    # --- 2. ADD CENTRAL HUB ---
    net.add_node(hub_name, label=hub_name, title="Central Hub: " + hub_name, color="#FFD700", size=40)

    # --- 3. ADD CATEGORY SATELLITES & LEAVES ---
    
    # Pillar A: Spatial (Locations)
    loc_cat = "Locations"
    net.add_node(loc_cat, label=loc_cat, color="#00BFFF", size=25)
    net.add_edge(hub_name, loc_cat, weight=5)
    
    for loc in satellites.get("spatial_pillar", []):
        net.add_node(loc, label=loc, title=f"Location: {loc}", color="#87CEEB", size=15)
        net.add_edge(loc_cat, loc)

    # Pillar B: Attributes (Traits)
    attr_cat = "Attributes"
    net.add_node(attr_cat, label=attr_cat, color="#32CD32", size=25)
    net.add_edge(hub_name, attr_cat, weight=5)
    
    for attr in satellites.get("attribute_pillar", []):
        net.add_node(attr, label=attr, title=f"Trait: {attr}", color="#90EE90", size=15)
        net.add_edge(attr_cat, attr)

    # Pillar C: Causal Timeline (Macro-Events)
    event_cat = "Causal Timeline"
    net.add_node(event_cat, label=event_cat, color="#FF4500", size=25)
    net.add_edge(hub_name, event_cat, weight=5)
    
    # Track event order to show causal sequence
    prev_event = event_cat
    for event in satellites.get("causal_pillar", []):
        e_name = event["event_name"]
        e_summary = event["summary"]
        # Use HTML for rich tooltips (Summary + Evidence Stack)
        e_title = f"<b>{e_name}</b><br><br><i>Summary:</i> {e_summary}<br><br><i>Evidence:</i><br>• " + "<br>• ".join(event["evidence_stack"][:5])
        
        net.add_node(e_name, label=e_name, title=e_title, color="#FA8072", size=20)
        # Primary link to category
        net.add_edge(event_cat, e_name)
        # Sequential link (Causal Flow)
        if prev_event != event_cat:
            net.add_edge(prev_event, e_name, label="Next Event", color="#555555", arrows="to")
        prev_event = e_name

    # --- 4. PHYSICS & LAYOUT ---
    net.force_atlas_2based() # Specialized for star/cluster layouts
    net.show_buttons(filter_=['physics']) # Allow UI adjustment of physics

    # --- 5. SAVE ---
    net.save_graph(OUTPUT_FILE)
    print(f"Visualization complete. Star Graph saved at: {OUTPUT_FILE}")

# RUN
visualize_star_graph()